# Explore the openmed-deidentification Cloud Run endpoint

Hits the deployed service directly over HTTP -- no `openmed` package install needed here, just `httpx`. Covers: health checks, a single de-identify call, PII extraction, a small batch loop, and a pointer to the interactive `/docs` page (open, no API key needed).

The API key is never hardcoded below -- it's prompted for with `getpass` so it doesn't end up committed in this notebook's output.

In [ ]:
# !pip install httpx

In [ ]:
import getpass
import httpx

BASE_URL = "https://openmed-deidentification-628896201179.us-central1.run.app"
MODEL = "OpenMed/privacy-filter-multilingual-v2"

API_KEY = getpass.getpass("OPENMED_API_KEY: ")

client = httpx.Client(
    base_url=BASE_URL,
    headers={"X-API-Key": API_KEY},
    timeout=180.0,  # CPU-only inference on this model can take 40+ seconds per request
)
print(f"Configured client for {BASE_URL}")

## Health checks

`/health` and `/readyz` don't require the API key -- neither does `/docs` if you want to browse the interactive schema in a browser: [{BASE_URL}/docs]

In [ ]:
print(client.get("/health").json())
print(client.get("/readyz").json())
print(f"Interactive docs: {BASE_URL}/docs")

## Single de-identify request

Expect this to take roughly 30-50 seconds -- CPU-only inference for a 1.4B-parameter MoE model, not a hung request. See the latency note in the repo README.

In [ ]:
text = "Patient Jordan Ramirez, MRN 4482910, called from 555-0147 about a refill."

response = client.post(
    "/pii/deidentify",
    json={"text": text, "method": "mask", "lang": "en", "model_name": MODEL},
)
response.raise_for_status()
result = response.json()

print("Original:     ", result["original_text"])
print("De-identified:", result["deidentified_text"])
print("Entities redacted:", result["num_entities_redacted"])

## PII extraction (entities + spans, no redaction)

In [ ]:
response = client.post(
    "/pii/extract",
    json={"text": text, "lang": "en", "use_smart_merging": True, "model_name": MODEL},
)
response.raise_for_status()

for entity in response.json()["entities"]:
    print(f"{entity['label']:<15} [{entity['start']:>3}:{entity['end']:<3}]  {entity['text']!r}  conf={entity['confidence']:.3f}")

## Small batch loop

For anything larger than a handful of documents, use `client/batch_client.py` from the repo root instead -- concurrent, retried, writes results incrementally to a JSONL file. This cell is just for quick exploration in the notebook.

Each request is slow (see above), so a handful of texts here can take a couple of minutes sequentially.

In [ ]:
sample_texts = [
    "Patient Jordan Ramirez, MRN 4482910, called from 555-0147.",
    "Paciente: Maria Garcia, correo maria.garcia@example.com, tel 555-0199.",
]

for i, sample in enumerate(sample_texts, start=1):
    response = client.post(
        "/pii/deidentify",
        json={"text": sample, "method": "mask", "model_name": MODEL},
    )
    response.raise_for_status()
    print(f"[{i}/{len(sample_texts)}] {response.json()['deidentified_text']}")

## Next steps

- Real batch processing: `python ../client/batch_client.py --base-url {BASE_URL} --api-key $OPENMED_API_KEY --input ../examples/sample_input.jsonl --output results.jsonl`
- Throughput benchmarking against this exact endpoint: `python ../client/benchmark_rest.py --base-url {BASE_URL} --api-key $OPENMED_API_KEY --concurrency-levels 1,2,4,8`
- Full endpoint reference: `{BASE_URL}/docs`